<a href="https://colab.research.google.com/github/mld8n13k/Machine-Learning-2026/blob/main/notebooks/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа 1. Разведочный анализ данных и классификация

## 0. Описание задачи
* **Датасет:** Hotel Reservations Dataset (Kaggle).
* **Цель:** Предсказать статус бронирования отеля (`booking_status`: отменено или подтверждено).
* **Физический смысл:** Помочь гостинице заранее выявлять бронирования с высоким риском отмены для оптимизации овербукинга и предотвращения финансовых потерь.
* **Тип задачи:** Бинарная классификация ($K = 2$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Читаем загруженный CSV файл
df = pd.read_csv('hotel_reservations.csv')

# Удаляем идентификатор брони — он не является физическим признаком
if 'Booking_ID' in df.columns:
    df = df.drop(columns=['Booking_ID'])

print(f"Размерность датасета: {df.shape}")
df.head()

In [ ]:
TARGET = 'booking_status'

# Преобразуем целевую переменную в числа: 0 - Not_Canceled, 1 - Canceled
df[TARGET] = (df[TARGET] == 'Canceled').astype(int)

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Разбиваем на train (75%) и test (25%) со стратификацией по классам
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape}, Тестовая: {X_test.shape}")
print("\nБаланс классов в Train:")
print(y_train.value_counts(normalize=True))

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Корреляционная матрица числовых признаков
plt.figure(figsize=(10, 7))
sns.heatmap(X_train[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Корреляционная матрица")
plt.show()

# Распределение классов
plt.figure(figsize=(5, 3))
sns.countplot(x=y_train)
plt.title("Распределение целевого класса (0 = Not_Canceled, 1 = Canceled)")
plt.show()

In [ ]:
# Визуализация взаимного распределения ключевых числовых признаков с раскраской по классам
key_features = ['lead_time', 'avg_price_per_room', 'no_of_special_requests']
sample_df = X_train[key_features].copy()
sample_df['target'] = y_train.map({0: 'Not Canceled', 1: 'Canceled'})

sns.pairplot(sample_df.sample(n=1000, random_state=42), hue='target', palette='Set1', diag_kind='kde')
plt.suptitle("Матрица рассеяния ключевых признаков", y=1.02)
plt.show()

In [ ]:
# Гистограммы всех числовых признаков
X_train[num_cols].hist(figsize=(14, 10), bins=25, edgecolor='black', color='steelblue')
plt.tight_layout()
plt.suptitle("Гистограммы числовых признаков выборки", y=1.02, fontsize=14)
plt.show()

In [ ]:
# 1. Проверка пропусков
print(f"Пропусков в train: {X_train.isnull().sum().sum()}")
print(f"Пропусков в test:  {X_test.isnull().sum().sum()}")

# 2. Оценка выбросов по межквартильному размаху (IQR)
for col in ['lead_time', 'avg_price_per_room']:
    q25 = X_train[col].quantile(0.25)
    q75 = X_train[col].quantile(0.75)
    iqr = q75 - q25
    upper_bound = q75 + 1.5 * iqr
    outliers = (X_train[col] > upper_bound).sum()
    print(f"Признак '{col}': {outliers} объектов выше {upper_bound:.2f} ({outliers / len(X_train) * 100:.2f}%)")
# Выбросы сохраняем, так как долгосрочные бронирования и люксы — реальные данные, а не ошибка.

In [ ]:
# Кодирование категориальных признаков
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Масштабирование признаков (критично для k-NN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Масштабирование StandardScaler успешно выполнено.")

In [ ]:
# Поиск k по сетке
param_grid = {'n_neighbors': [3, 5, 7, 9, 11]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

best_k = grid.best_params_['n_neighbors']
print(f"Оптимальное k: {best_k}")

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

y_pred_train = knn.predict(X_train_scaled)
y_pred_test = knn.predict(X_test_scaled)

print(f"Accuracy Train: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Accuracy Test:  {accuracy_score(y_test, y_pred_test):.4f}")

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Canceled', 'Canceled'],
            yticklabels=['Not Canceled', 'Canceled'])
plt.title(f"Матрица рассогласования (k={best_k})")
plt.xlabel("Предсказание")
plt.ylabel("Факт")
plt.show()

print("\nОтчет классификации:\n", classification_report(y_test, y_pred_test))

In [ ]:
# Оценка ошибки на обучении и тесте при разных k
k_range = [1, 3, 5, 7, 9, 13, 17, 21]
train_errors = []
test_errors = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_errors.append(1 - accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_errors.append(1 - accuracy_score(y_test, knn.predict(X_test_scaled)))

plt.figure(figsize=(8, 4))
plt.plot(k_range, train_errors, label='Ошибка на обучении (Train Error)', marker='o')
plt.plot(k_range, test_errors, label='Ошибка на тесте (Test Error)', marker='s')
plt.xlabel('Количество соседей k')
plt.ylabel('Доля ошибок (Error rate)')
plt.title('Зависимость ошибки классификатора k-NN от k')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Важность признаков по версии Random Forest
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.head(10).plot(kind='barh', color='teal')
plt.gca().invert_yaxis()
plt.title("Топ-10 наиболее информативных признаков (Random Forest)")
plt.xlabel("Важность признака")
plt.show()

In [ ]:
# Обучение Random Forest со сбалансированными весами классов
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nОтчет классификации Random Forest:\n", classification_report(y_test, y_pred_rf))

## 13. Общие выводы
1. **Данные:** Пропущенные значения отсутствуют. Выбросы в `lead_time` и `avg_price_per_room` отражают реальные бизнес-процессы (ранняя покупка туров, бронь люксов) и оставлены в выборке.
2. **Масштабирование:** Стандартизация признаков позволила метрическому классификатору корректно учитывать разнородные признаки (валюту, интервалы времени и бинарные индикаторы).
3. **Результаты моделей:**
   * $k$-NN показал точность около ~85%.
   * Random Forest показал результат лучше (~89–90%) и точнее выявляет класс отмен благодаря учету нелинейных взаимосвязей и параметру `class_weight='balanced'`.